In [2]:
import nltk
import ssl

# Eğer Windows'ta veya ağda SSL sertifika engeli varsa aşmak için:
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Modelleri eksiksiz indirmesi için zorluyoruz
print("İndirmeler başlıyor...")
nltk.download('punkt', force=True)
nltk.download('punkt_tab', force=True)  # NLTK'nin güncel sürümleri tokenization için bu paketi de arar!
nltk.download('stopwords', force=True)
nltk.download('wordnet', force=True)
print("İndirmeler tamamlandı!")

İndirmeler başlıyor...


[nltk_data] Downloading package punkt to C:\Users\Zeki
[nltk_data]     Kurt\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to C:\Users\Zeki
[nltk_data]     Kurt\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to C:\Users\Zeki
[nltk_data]     Kurt\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to C:\Users\Zeki
[nltk_data]     Kurt\AppData\Roaming\nltk_data...


İndirmeler tamamlandı!


In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


df = pd.read_csv('../data/processed/cleaned_reviews.csv')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


sample_text = "The batteries are REALLY terrible! I bought them for my Kindle, but they died quickly."
print("0. Orijinal Metin:\n", sample_text, "\n")

# Adım 1: Lowercasing ve Noktalama Temizliği

text_lower = re.sub(r'[^\w\s]', '', sample_text.lower())
print("1. Lowercasing ve Noktalama Temizliği Sonrası:\n", text_lower, "\n")

# Adım 2: Tokenization ve Stop Word Temizliği
# Bağlaçlar (the, are, for, but vb.) cümlenin duygu durumuna etki etmediği için çıkarılır
tokens = word_tokenize(text_lower)
text_no_stop = [word for word in tokens if word not in stop_words]
print("2. Stop Word Temizliği Sonrası:\n", text_no_stop, "\n")

# Adım 3: Lemmatization
# Kelimeleri köklerine indirgeriz (örn: batteries -> battery)
text_lemmatized = [lemmatizer.lemmatize(word) for word in text_no_stop]
print("3. Lemmatization Sonrası:\n", text_lemmatized, "\n")

print("-> Sonuç Cümlesi:\n", " ".join(text_lemmatized))

[nltk_data] Downloading package punkt to C:\Users\Zeki
[nltk_data]     Kurt\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Zeki
[nltk_data]     Kurt\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Zeki
[nltk_data]     Kurt\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


0. Orijinal Metin:
 The batteries are REALLY terrible! I bought them for my Kindle, but they died quickly. 

1. Lowercasing ve Noktalama Temizliği Sonrası:
 the batteries are really terrible i bought them for my kindle but they died quickly 

2. Stop Word Temizliği Sonrası:
 ['batteries', 'really', 'terrible', 'bought', 'kindle', 'died', 'quickly'] 

3. Lemmatization Sonrası:
 ['battery', 'really', 'terrible', 'bought', 'kindle', 'died', 'quickly'] 

-> Sonuç Cümlesi:
 battery really terrible bought kindle died quickly


In [4]:
# Tüm veri seti için ön işleme fonksiyonu
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', '', str(text).lower())
    tokens = word_tokenize(text)
    processed = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(processed)

# İşlemi uygula ve yeni sütun olarak ekle
df['processed_text'] = df['text'].apply(preprocess_text)

# Eksik veya tamamen boşalan satırları (sadece stop-word'den oluşan yorumları) temizle
df.dropna(subset=['processed_text'], inplace=True)
df = df[df['processed_text'].str.strip() != '']

# Modellerde kullanılmak üzere kaydedelim
df.to_csv('../data/processed/nlp_processed_reviews.csv', index=False)
print("Metin ön işleme tamamlandı ve 'nlp_processed_reviews.csv' olarak kaydedildi.")

Metin ön işleme tamamlandı ve 'nlp_processed_reviews.csv' olarak kaydedildi.


Metin Vektörizasyon Tercihi: TF-IDF vs Bag-of-Words (BoW)

Bag-of-Words (BoW): Metindeki kelimelerin corpus içerisinde kaç kez geçtiğini (frekansını) sayarak bir matris oluşturur. Sık geçen kelimelere yüksek ağırlık verir. Ancak, metnin bağlamını veya o kelimenin ilgili dokümanı diğerlerinden ayırmadaki gücünü dikkate almaz.

TF-IDF (Term Frequency-Inverse Document Frequency): Bir kelimenin ilgili yorumda ne kadar sık geçtiğini (TF) ölçerken, tüm yorum setinde  ne kadar yaygın olduğunu (IDF) hesaplayıp bu değeri normalize eder.

Tercih Gerekçesi (TF-IDF): E-ticaret yorumlarında "ürün, kindle, ekran, cihaz" gibi isim kökenli kelimeler çok sık geçer ancak olumlu/olumsuz bir duygu (sentiment) belirtmezler. BoW kullanıldığında model bu nötr kelimelere gereksiz bir ağırlık atfeder. TF-IDF ise corpus genelinde çok sık geçen bu kelimeleri cezalandırarak (ağırlığını düşürerek); "berbat, harika, donuyor, kusursuz" gibi nadir ama duygu sınıflandırması için ayırt edici olan (discriminative) kelimeleri öne çıkarır. Bu nedenle projedeki duygu analizi modellemesinde TF-IDF yöntemi kullanılacaktır.